<a href="https://colab.research.google.com/github/yugethayuva-debug/language-detector/blob/main/Copy_of_Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf



In [ ]:
!pip install kagglehub -q
import kagglehub, os, pandas as pd

path = kagglehub.dataset_download("zarajamshaid/language-identification-datasst")
print(os.listdir(path))

100%|██████████| 5.53M/5.53M [00:00<00:00, 144MB/s]

Extracting files...
['dataset.csv']


In [ ]:
df = pd.read_csv(os.path.join(path, "dataset.csv"))
print(df.head())
print(df.columns)
print(df.shape)

                                                Text  language
0  klement gottwaldi surnukeha palsameeriti ning ...  Estonian
1  sebes joseph pereira thomas  på eng the jesuit...   Swedish
2  ถนนเจริญกรุง อักษรโรมัน thanon charoen krung เ...      Thai
3  விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...     Tamil
4  de spons behoort tot het geslacht haliclona en...     Dutch
Index(['Text', 'language'], dtype='object')
(22000, 2)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print(df["language"].nunique(), "languages")
print(df.isnull().sum())

le = LabelEncoder()
y = le.fit_transform(df["language"])

X_train, X_test, y_train, y_test = train_test_split(
    df["Text"].astype(str), y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)

22 languages
Text        0
language    0
dtype: int64
(17600,) (4400,)


In [ ]:
from tensorflow.keras import layers, models

vectorizer = layers.TextVectorization(
    max_tokens=5000, output_sequence_length=200, split="character")
vectorizer.adapt(X_train.values)

model = models.Sequential([
    vectorizer,
    layers.Embedding(5000, 64),
    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(len(le.classes_), activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model = models.Sequential([
    vectorizer,
    layers.Embedding(5000, 64, mask_zero=True),
    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(len(le.classes_), activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam", metrics=["accuracy"])
model.fit(X_train.values, y_train, epochs=10, batch_size=64,
          validation_data=(X_test.values, y_test))

Epoch 1/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.6890 - loss: 1.3609 - val_accuracy: 0.9016 - val_loss: 0.5925
Epoch 2/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - accuracy: 0.9244 - loss: 0.3757 - val_accuracy: 0.9359 - val_loss: 0.2759
Epoch 3/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9470 - loss: 0.2342 - val_accuracy: 0.9473 - val_loss: 0.2250
Epoch 4/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.9546 - loss: 0.1959 - val_accuracy: 0.9548 - val_loss: 0.1953
Epoch 5/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - accuracy: 0.9584 - loss: 0.1776 - val_accuracy: 0.9582 - val_loss: 0.1817
Epoch 6/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9618 - loss: 0.1648 - val_accuracy: 0.9627 - val_loss: 0.1754
Epoch 7/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - accuracy: 0.9628 - loss: 0.1577 - val_accuracy: 0.9611 - val_loss: 0.1687
Epoch 8/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.9643 - loss: 0.1519 - val_ac

In [ ]:
print(detect("Bonjour, comment allez-vous aujourd'hui? J'espère que tout va bien pour vous."))
print(detect("How are you doing today? I hope everything is going well for you."))

French
English


In [ ]:
from sklearn.metrics import classification_report
pred = model.predict(X_test.values, verbose=0).argmax(axis=1)
print(classification_report(y_test, pred, target_names=le.classes_))

              precision    recall  f1-score   support

      Arabic       1.00      1.00      1.00       200
     Chinese       0.99      0.99      0.99       200
       Dutch       0.94      0.94      0.94       200
     English       0.77      0.90      0.83       200
    Estonian       0.96      0.94      0.95       200
      French       0.94      0.98      0.96       200
       Hindi       1.00      0.96      0.98       200
  Indonesian       0.97      0.95      0.96       200
    Japanese       1.00      0.99      0.99       200
      Korean       1.00      0.99      1.00       200
       Latin       0.87      0.91      0.89       200
     Persian       0.99      0.99      0.99       200
   Portugese       0.95      0.91      0.93       200
      Pushto       1.00      0.94      0.97       200
    Romanian       0.98      0.97      0.98       200
     Russian       0.99      0.99      0.99       200
     Spanish       0.95      0.95      0.95       200
     Swedish       0.98    

In [ ]:
model.save("language_detector.keras")

In [ ]:
from tensorflow.keras import layers, models

lstm_model = models.Sequential([
    vectorizer,
    layers.Embedding(5000, 64, mask_zero=True),
    layers.LSTM(64),
    layers.Dense(128, activation="relu"),
    layers.Dense(len(le.classes_), activation="softmax")
])

lstm_model.compile(loss="sparse_categorical_crossentropy",
                   optimizer="adam", metrics=["accuracy"])
lstm_model.fit(X_train.values, y_train, epochs=5, batch_size=64,
               validation_data=(X_test.values, y_test))

Epoch 1/5
275/275 ━━━━━━━━━━━━━━━━━━━━ 53s 184ms/step - accuracy: 0.3940 - loss: 1.7966 - val_accuracy: 0.5693 - val_loss: 1.2945
Epoch 2/5
275/275 ━━━━━━━━━━━━━━━━━━━━ 82s 185ms/step - accuracy: 0.5826 - loss: 1.1964 - val_accuracy: 0.4400 - val_loss: 1.5755
Epoch 3/5
275/275 ━━━━━━━━━━━━━━━━━━━━ 50s 183ms/step - accuracy: 0.5911 - loss: 1.1954 - val_accuracy: 0.6380 - val_loss: 0.9784
Epoch 4/5
275/275 ━━━━━━━━━━━━━━━━━━━━ 84s 190ms/step - accuracy: 0.6624 - loss: 0.9622 - val_accuracy: 0.7002 - val_loss: 0.8475
Epoch 5/5
275/275 ━━━━━━━━━━━━━━━━━━━━ 50s 182ms/step - accuracy: 0.7250 - loss: 0.7524 - val_accuracy: 0.7789 - val_loss: 0.6237


In [ ]:
loss, acc = lstm_model.evaluate(X_test.values, y_test)
print("LSTM Test accuracy:", acc)

138/138 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.7789 - loss: 0.6237
LSTM Test accuracy: 0.7788636088371277


In [ ]:
lstm_model.fit(X_train.values, y_train, epochs=10, batch_size=64,
               validation_data=(X_test.values, y_test))

Epoch 1/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 50s 184ms/step - accuracy: 0.7901 - loss: 0.5765 - val_accuracy: 0.8291 - val_loss: 0.4822
Epoch 2/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 49s 178ms/step - accuracy: 0.8511 - loss: 0.4297 - val_accuracy: 0.8632 - val_loss: 0.3529
Epoch 3/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 52s 188ms/step - accuracy: 0.8625 - loss: 0.4075 - val_accuracy: 0.8839 - val_loss: 0.3287
Epoch 4/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 52s 188ms/step - accuracy: 0.9139 - loss: 0.2904 - val_accuracy: 0.9193 - val_loss: 0.2793
Epoch 5/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 80s 182ms/step - accuracy: 0.9321 - loss: 0.2490 - val_accuracy: 0.9368 - val_loss: 0.2328
Epoch 6/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 50s 184ms/step - accuracy: 0.9415 - loss: 0.2236 - val_accuracy: 0.9380 - val_loss: 0.2312
Epoch 7/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 81s 180ms/step - accuracy: 0.9472 - loss: 0.1989 - val_accuracy: 0.9509 - val_loss: 0.1914
Epoch 8/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 52s 188ms/step - accuracy: 0.9538 - loss: 0